# 04. OCR Experiments: Images, GIFs & Videos (`OcrExperimentAdapter`)

In [1]:
import json
from pathlib import Path

import pandas as pd

from meme_organizer.config import CYRILLIC_OCR_DICT_PATH, CYRILLIC_OCR_MODEL_PATH, MLFLOW_EXPERIMENT_OCR
from meme_organizer.evaluation.metrics.experiment_tracker import ExperimentTracker
from meme_organizer.extraction.ocr import OcrPreprocessor, RapidOcrExtractor
from meme_organizer.extraction.ocr.ocr_experiment_adapter import OcrExperimentAdapter
from meme_organizer.extraction.video import KeyframeSampler

ROOT = Path("..").resolve()
GOLD_PATH = ROOT / "data" / "gold" / "gold_dataset.jsonl"
SILVER_DIR = ROOT / "data" / "silver"

records = [json.loads(line) for line in GOLD_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
img_records = [r for r in records if r["media_type"] == "image" and r.get("transcript_or_text", "").strip()]
vid_records = [r for r in records if r["media_type"] != "image" and r.get("transcript_or_text", "").strip()]
print(f"Images: {len(img_records)} | Videos/GIFs: {len(vid_records)}")

rapid_cyr = RapidOcrExtractor(
    min_confidence=0.5,
    rec_model_path=str(ROOT / CYRILLIC_OCR_MODEL_PATH),
    rec_keys_path=str(ROOT / CYRILLIC_OCR_DICT_PATH),
)
results = []

2026/11/27 17:33:28 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/alex/Projects/meme-organizer/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Images: 124 | Videos/GIFs: 115


In [2]:
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_OCR) as exp:
    adapter = OcrExperimentAdapter(
        run_name="img_rapidocr_cyrillic_raw",
        extractor=rapid_cyr,
        preprocessor=OcrPreprocessor(mode="raw"),
        sampler=KeyframeSampler(strategy="uniform_3"),
        records=img_records,
        silver_dir=SILVER_DIR,
        rec_model_path=ROOT / CYRILLIC_OCR_MODEL_PATH,
        rec_keys_path=ROOT / CYRILLIC_OCR_DICT_PATH,
    )
    res = exp.create_pipeline(adapter).run()
results.append({"run": "img_raw", **res["quality"], **res["performance"]})
print("Images:", res["quality"], res["performance"])

🏃 View run img_rapidocr_cyrillic_raw at: http://localhost:5000/#/experiments/11/runs/337005cc75544034b9a57e7c94d3b5a5
🧪 View experiment at: http://localhost:5000/#/experiments/11
Images: {'cer': 0.2186, 'wer': 0.4042, 'empty_rate': 0.0242, 'avg_confidence': 0.8913} {'sample_p50_ms': 5.66, 'sample_p95_ms': 12.77, 'preprocess_p50_ms': 0.0, 'inference_p50_ms': 446.0, 'inference_p95_ms': 920.75, 'total_p50_ms': 452.27, 'total_p95_ms': 934.99, 'throughput_items_per_sec': 1.99, 'model_size_mb': 15.5, 'total_pipeline_time_sec': 62.415}


In [3]:
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_OCR) as exp:
    adapter = OcrExperimentAdapter(
        run_name="vid_rapidocr_cyrillic_uniform3",
        extractor=rapid_cyr,
        preprocessor=OcrPreprocessor(mode="raw"),
        sampler=KeyframeSampler(strategy="uniform_3"),
        records=vid_records,
        silver_dir=SILVER_DIR,
        rec_model_path=ROOT / CYRILLIC_OCR_MODEL_PATH,
        rec_keys_path=ROOT / CYRILLIC_OCR_DICT_PATH,
    )
    res = exp.create_pipeline(adapter).run()
results.append({"run": "vid_uniform3", **res["quality"], **res["performance"]})
print("Video uniform_3:", res["quality"], res["performance"])

🏃 View run vid_rapidocr_cyrillic_uniform3 at: http://localhost:5000/#/experiments/11/runs/84b0a3e1b6844c3abf5f34aa400313f7
🧪 View experiment at: http://localhost:5000/#/experiments/11
Video uniform_3: {'cer': 0.8726, 'wer': 0.9318, 'empty_rate': 0.4174, 'avg_confidence': 0.9184} {'sample_p50_ms': 226.06, 'sample_p95_ms': 513.12, 'preprocess_p50_ms': 0.0, 'inference_p50_ms': 1240.73, 'inference_p95_ms': 1987.94, 'total_p50_ms': 1481.1, 'total_p95_ms': 2297.31, 'throughput_items_per_sec': 0.65, 'model_size_mb': 15.5, 'total_pipeline_time_sec': 175.988}


In [4]:
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_OCR) as exp:
    adapter = OcrExperimentAdapter(
        run_name="vid_rapidocr_cyrillic_interval_dedup",
        extractor=rapid_cyr,
        preprocessor=OcrPreprocessor(mode="raw"),
        sampler=KeyframeSampler(strategy="interval_dedup"),
        records=vid_records,
        silver_dir=SILVER_DIR,
        rec_model_path=ROOT / CYRILLIC_OCR_MODEL_PATH,
        rec_keys_path=ROOT / CYRILLIC_OCR_DICT_PATH,
    )
    res = exp.create_pipeline(adapter).run()
results.append({"run": "vid_interval_dedup", **res["quality"], **res["performance"]})
print("Video interval_dedup:", res["quality"], res["performance"])
print(
    pd.DataFrame(results)[
        ["run", "cer", "wer", "empty_rate", "avg_confidence", "sample_p50_ms", "inference_p50_ms", "total_p50_ms"]
    ].to_string(index=False)
)

🏃 View run vid_rapidocr_cyrillic_interval_dedup at: http://localhost:5000/#/experiments/11/runs/3b52b347594e4b9c8714c2f40ddf77d8
🧪 View experiment at: http://localhost:5000/#/experiments/11
Video interval_dedup: {'cer': 0.8965, 'wer': 0.9426, 'empty_rate': 0.4609, 'avg_confidence': 0.9089} {'sample_p50_ms': 489.62, 'sample_p95_ms': 977.3, 'preprocess_p50_ms': 0.0, 'inference_p50_ms': 897.41, 'inference_p95_ms': 3415.43, 'total_p50_ms': 1392.36, 'total_p95_ms': 4400.65, 'throughput_items_per_sec': 0.54, 'model_size_mb': 15.5, 'total_pipeline_time_sec': 213.321}
               run    cer    wer  empty_rate  avg_confidence  sample_p50_ms  inference_p50_ms  total_p50_ms
           img_raw 0.2186 0.4042      0.0242          0.8913           5.66            446.00        452.27
      vid_uniform3 0.8726 0.9318      0.4174          0.9184         226.06           1240.73       1481.10
vid_interval_dedup 0.8965 0.9426      0.4609          0.9089         489.62            897.41       1392.36
